# Task 1 -- clean run

**Before running anything:**
1. Notebook Settings -> Accelerator: GPU, Internet: On.
2. Add Input -> Upload -> `task1_pa1_v3_fixed.zip` (the bug-fixed code). Any dataset title works as long as it's 6+ characters.

Then just run every cell below in order, top to bottom. No manual patching needed this time -- all four bugs found during the last run (ProbeConfig kwarg mismatch, TSNE `n_iter`/`max_iter` rename, CLIP tokenizer device mismatch, patch-shuffle permutation device mismatch) are already fixed in this zip.

In [ ]:
# Sanity check: GPU + internet must both work before anything else will.
!nvidia-smi
!curl -sI https://www.google.com | head -1

In [ ]:
# Auto-locate the uploaded dataset under /kaggle/input, regardless of
# whatever username/slug Kaggle gave it -- avoids the path-guessing pain
# from last time.
import glob, os

candidates = glob.glob("/kaggle/input/**/run_task1.py", recursive=True)
assert candidates, (
    "Could not find run_task1.py under /kaggle/input -- "
    "did you Add Input -> Upload the zip and attach it to this notebook?"
)
task1_src = os.path.dirname(os.path.dirname(candidates[0]))
print("Found source at:", task1_src)

In [ ]:
# Copy into the writable working directory (Kaggle can't write into /kaggle/input),
# then install the two packages Kaggle doesn't ship by default.
import shutil

if os.path.exists("/kaggle/working/task1"):
    shutil.rmtree("/kaggle/working/task1")
shutil.copytree(task1_src, "/kaggle/working/task1")
%cd /kaggle/working/task1
!pip install -q open_clip_torch umap-learn

In [ ]:
# Download the AdaIN decoder weights (needed for the Step 3 cue-conflict images).
!mkdir -p checkpoints
!wget -q https://github.com/naoto0804/pytorch-AdaIN/releases/download/v0.0.0/decoder.pth -O checkpoints/decoder.pth
!ls -la checkpoints/

In [ ]:
# Point the dataset root at somewhere writable, and confirm we're using
# the real (not dry-run) numbers.
import yaml

cfg = yaml.safe_load(open("configs/config.yaml"))
cfg["dataset"]["root"] = "/kaggle/working/data/raw"
cfg["test_subset"]["size"] = 500
cfg["interventions"]["cue_conflict"]["min_valid_conflicts"] = 200
cfg["interventions"]["cue_conflict"]["images_per_pair_per_direction"] = 24
cfg["linear_probe"]["max_epochs"] = 50
yaml.safe_dump(cfg, open("configs/config.yaml", "w"))
print(cfg["dataset"])
print(cfg["test_subset"])
print(cfg["linear_probe"])

In [ ]:
# Downloads Oxford-IIIT Pets (~800MB) and builds the reproducible splits.
!python data/make_subset.py --config configs/config.yaml

In [ ]:
# Generates the shape/texture cue-conflict images.
!python data/make_cue_conflicts.py --config configs/config.yaml

In [ ]:
# The big one: trains all 3 backbones' heads and runs every required intervention.
!python scripts/run_task1.py --config configs/config.yaml

In [ ]:
# Zip up the ACTUAL final results (run this only after the cell above finishes),
# then download task1_results.zip from the Output panel on the right.
!rm -f /kaggle/working/task1_results.zip
!zip -r /kaggle/working/task1_results.zip results/
!unzip -l /kaggle/working/task1_results.zip